# K-IFRS 4-Step 검색 파이프라인 테스트

| Step | 설명 | 데이터 소스 |
|------|------|------------|
| 1 | 기준서 식별 | `standard_summaries.embedding` |
| 2 | Level 1 문단 검색 | `chunks` (authority=1) |
| 3 | IE 적용사례 | `paragraph_links` (source_component='ie') |
| 4 | BC 결론도출근거 | `paragraph_links` (source_component='bc') |

In [1]:
# DB 연결 + Embedder 초기화
import psycopg
from pgvector.psycopg import register_vector
from dotenv import load_dotenv
load_dotenv()

from ingester.embedder import Embedder

conn = psycopg.connect("dbname=kifrs", autocommit=True)
register_vector(conn)
embedder = Embedder()

print("DB 연결 OK")
print(f"Embedding model: {embedder.model}")

DB 연결 OK
Embedding model: embedding-passage


## Step 1: 기준서 식별
쿼리를 임베딩 → `standard_summaries` 테이블에서 코사인 유사도 top-5

In [2]:
def step1_identify_standard(query: str, top_k: int = 5):
    """Step 1: 쿼리에 가장 적합한 기준서 식별"""
    query_emb = embedder.embed_single(query)
    
    rows = conn.execute("""
        SELECT standard_id, title, 
               1 - (embedding <=> %s::vector) AS similarity
        FROM standard_summaries
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (query_emb, query_emb, top_k)).fetchall()
    
    print(f"Query: \"{query}\"\n")
    print(f"{'순위':<4} {'유사도':<8} {'기준서':<20} {'제목'}")
    print("-" * 70)
    for i, (sid, title, sim) in enumerate(rows, 1):
        print(f"{i:<4} {sim:.4f}   {sid:<20} {title}")
    
    return rows

# 테스트
results = step1_identify_standard("수행의무 판단 기준")

Query: "수행의무 판단 기준"

순위   유사도      기준서                  제목
----------------------------------------------------------------------
1    0.4305   K-IFRS 2121          부담금
2    0.4259   K-IFRS 2123          법인세 처리의 불확실성
3    0.4253   K-IFRS 2110          중간재무보고와 손상
4    0.4199   K-IFRS 1111          공동약정
5    0.4183   K-IFRS 2107          제1029호 '초인플레이션 경제에서의 재무보고'에 따른 재작성 방법의 적용


## Step 2: Level 1 문단 검색
선택된 기준서 내에서 authority=1 청크를 벡터 검색. main → ag 순서로 그룹핑.

In [3]:
COMPONENT_ORDER = {"main": 0, "definitions": 1, "ag": 2, "transition": 3}

def step2_search_authoritative(query: str, standard_id: str, top_k: int = 10):
    """Step 2: 기준서 내 Level 1 문단 벡터 검색"""
    query_emb = embedder.embed_single(query)
    
    rows = conn.execute("""
        SELECT chunk_id, para_number, component, section_title,
               content_markdown,
               1 - (embedding <=> %s::vector) AS similarity
        FROM chunks
        WHERE standard_id = %s AND authority = 1
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (query_emb, standard_id, query_emb, top_k)).fetchall()
    
    # component 순서로 그룹핑
    rows_sorted = sorted(rows, key=lambda r: (COMPONENT_ORDER.get(r[2], 99), -r[5]))
    
    print(f"Query: \"{query}\" in {standard_id}\n")
    current_comp = None
    for chunk_id, para, comp, section, md, sim in rows_sorted:
        if comp != current_comp:
            comp_label = {"main": "본문", "ag": "적용지침", "definitions": "정의", "transition": "경과규정"}
            print(f"\n### {comp_label.get(comp, comp)} ({comp})")
            current_comp = comp
        preview = md[:150].replace('\n', ' ')
        print(f"  [{sim:.3f}] 문단 {para or 'N/A'} ({section or '-'})")
        print(f"         {preview}...")
    
    # 문단번호 목록 반환 (Steps 3-4에서 사용)
    para_numbers = [r[1] for r in rows if r[1]]
    return rows_sorted, para_numbers

# 테스트
chunks, para_nums = step2_search_authoritative("수행의무 판단 기준", "K-IFRS 1115")
print(f"\n검색된 문단번호: {para_nums}")

Query: "수행의무 판단 기준" in K-IFRS 1115


### 본문 (main)
  [0.557] 문단 B55 (공시)
         B55		라이선스가 구별되지 않는다면, 수행의무(약속한 라이선스 포함)가 기간에 걸쳐 이행되는 수행의무인지 한 시점에 이행되는 수행의무인지를 판단하기 위해 문단 31~38을 적용한다....
  [0.554] 문단 34 (인식)
         34		고객이 자산을 통제하는지를 판단할 때, 그 자산을 재매입하는 약정을 고려한다(문단 B64~B76 참조). **기간에 걸쳐 이행하는 수행의무**...
  [0.547] 문단 14 (인식)
         14		고객과의 계약이 문단 9의 기준을 충족하지 못한다면, 문단 9의 기준이 나중에 충족되는지를 판단하기 위해 그 계약을 지속적으로 검토한다....
  [0.546] 문단 32 (인식)
         32		문단 22~30에 따라 식별한 각 수행의무를 (문단 35~37에 따라) 기간에 걸쳐 이행하는지 또는 (문단 38에 따라) 한 시점에 이행하는지를 계약 개시시점에 판단한다. 수행의무가 기간에 걸쳐 이행되지 않는다면, 그 수행의무는 한 시점에 이행되는 것이다....
  [0.526] 문단 44 (인식)
         44		수행의무의 진행률을 합리적으로 측정할 수 있는 경우에만, 기간에 걸쳐 이행하는 수행의무에 대한 수익을 인식한다. 적절한 진행률 측정방법을 적용하는 데 필요한 신뢰할 수 있는 정보가 부족하다면 수행의무의 진행률을 합리적으로 측정할 수 없을 것이다....
  [0.509] 문단 45 (인식)
         45		어떤 상황(예: 계약 초기 단계)에서는 수행의무의 결과를 합리적으로 측정할 수 없으나, 수행의무를 이행하는 동안에 드는 원가는 회수될 것으로 예상한다. 그 상황에서는 수행의무의 결과를 합리적으로 측정할 수 있을 때까지 발생원가의 범위에서만 수익을 인식한다....
  [0.504] 문단 123 (공시)
         123		이 기준서를 적

## Step 3 & 4: IE 적용사례 / BC 결론도출근거
Step 2에서 찾은 문단번호를 `paragraph_links`에서 조회하여 관련 IE/BC 반환.

In [4]:
def step3_4_find_related(standard_id: str, para_numbers: list[str], component: str, top_k: int = 5):
    """Step 3/4: paragraph_links를 통해 관련 IE/BC 청크 조회"""
    if not para_numbers:
        print("문단번호 없음 — 건너뜀")
        return []
    
    # 문단번호 범위 매칭: target_para_start가 검색된 문단번호 중 하나와 일치
    placeholders = ",".join(["%s"] * len(para_numbers))
    rows = conn.execute(f"""
        SELECT DISTINCT c.chunk_id, c.para_number, c.section_title,
               c.content_markdown,
               pl.target_para_start, pl.target_para_end, pl.link_type
        FROM paragraph_links pl
        JOIN chunks c ON c.chunk_id = pl.source_chunk_id
        WHERE pl.standard_id = %s
          AND pl.source_component = %s
          AND pl.target_para_start IN ({placeholders})
        LIMIT %s
    """, (standard_id, component, *para_numbers, top_k)).fetchall()
    
    comp_label = "적용사례 (IE)" if component == "ie" else "결론도출근거 (BC)"
    print(f"\n## {comp_label} — {standard_id}")
    print(f"검색 기준 문단: {para_numbers}\n")
    
    if not rows:
        print("  (링크된 자료 없음 — 벡터 검색 폴백 필요)")
        return []
    
    for chunk_id, para, section, md, target_start, target_end, link_type in rows:
        target = f"{target_start}~{target_end}" if target_end else target_start
        preview = md[:200].replace('\n', ' ')
        print(f"  [{link_type}] {para or 'N/A'} → 본문 문단 {target}")
        print(f"    {section or ''}")
        print(f"    {preview}...")
        print()
    
    return rows

# Step 3: IE
ie_results = step3_4_find_related("K-IFRS 1115", para_nums, "ie")

# Step 4: BC
bc_results = step3_4_find_related("K-IFRS 1115", para_nums, "bc")


## 적용사례 (IE) — K-IFRS 1115
검색 기준 문단: ['B55', '34', '14', '32', '44', '45', '123', 'B10', '25', '89']

  [body_reference] IE10 → 본문 문단 14
    계약의 식별
    IE10		기업(병원)은 응급실의 비보험환자에게 의료용역을 제공한다. 기업은 과거에 이 환자에게 의료용역을 제공하지 않았지만 법령에 따라 모든 응급실 환자에게 의료용역을 제공하여야 한다. 병원에 도착하는 환자의 상태 때문에 기업은 용역을 즉시 제공해야 하므로 제공되는 의료용역에 대한 대가로 환자가 계약에 따른 의무를 수행하기로 확약하는지를 기업이 판단할 수...

  [body_reference] IE6 → 본문 문단 14
    계약의 식별
    IE6		기업회계기준서 제1115호 문단 9의 기준을 충족하지 못하기 때문에, 기업은 환불되지 않는 계약금 50,000원의 회계처리 결정에 기업회계기준서 제1115호의 문단 15∼16을 적용한다. 기업은 문단 15에서 기술한 사건 중 어느 하나도 일어나지 않은 것으로 본다. 즉 기업이 대부분의 대가를 받지도 못하였고 계약을 종료하지도 않았다. 따라서 문단 ...


## 결론도출근거 (BC) — K-IFRS 1115
검색 기준 문단: ['B55', '34', '14', '32', '44', '45', '123', 'B10', '25', '89']

  [body_reference] BC414V → 본문 문단 B55
    적용지침(문단 B2~B89)
    BC414V		문단 B55에서는 구별되지 않는(문단 27에 따름) 라이선스를 포함하는 수행의무가 한 시점에 이행되는지 아니면 기간에 걸쳐 이행되는지를 판단하기 위해 일반적인 수익인식모형(문단 31∼38)을 적용하도록 요구한다. IFRS 15가 공표된 이후로 일부 이해관계자는 기업의 약속의 성격 판단에 대한 라이선싱 지침을 라이선스와 그 밖의 재화나 용역을 ...

  [body_referenc

## 정의 + LLM 컨텍스트 조합
기준서의 정의 전체를 가져오고, 4단계 결과를 LLM에 보낼 컨텍스트로 포맷팅.

In [5]:
def build_llm_context(standard_id: str, query: str, 
                       main_chunks, ie_results=None, bc_results=None):
    """4단계 결과를 LLM 컨텍스트로 포맷팅"""
    
    # 정의 가져오기
    row = conn.execute("""
        SELECT title, definitions_text FROM standard_summaries
        WHERE standard_id = %s
    """, (standard_id,)).fetchone()
    title = row[0] if row else standard_id
    definitions = row[1] if row and row[1] else ""
    
    ctx = []
    ctx.append(f"# {standard_id} {title}")
    ctx.append(f"사용자 질문: {query}\n")
    
    # 정의
    if definitions:
        ctx.append("## 용어 정의 [참조]")
        ctx.append(definitions[:3000])  # 최대 3000자
        ctx.append("")
    
    # Step 2: 본문 + AG
    ctx.append("## 적용 문단 [Authoritative, Level 1]")
    for chunk_id, para, comp, section, md, sim in main_chunks:
        label = "본문" if comp == "main" else "적용지침"
        ctx.append(f"\n**문단 {para or 'N/A'}** ({label}, {section or '-'})")
        ctx.append(md[:500])
    ctx.append("")
    
    # Step 3: IE
    if ie_results:
        ctx.append("## 적용사례 [Non-authoritative, Level 4]")
        for chunk_id, para, section, md, ts, te, lt in ie_results:
            ctx.append(f"\n**{para or 'IE'}** ({section or '-'})")
            ctx.append(md[:500])
        ctx.append("")
    
    # Step 4: BC
    if bc_results:
        ctx.append("## 결론도출근거 [Non-authoritative, Level 4]")
        ctx.append("*주의: 결론도출근거는 기준서의 일부를 구성하지 않습니다. 본문과 충돌 시 본문이 우선합니다.*\n")
        for chunk_id, para, section, md, ts, te, lt in bc_results:
            ctx.append(f"\n**{para or 'BC'}** ({section or '-'})")
            ctx.append(md[:500])
    
    full_context = "\n".join(ctx)
    print(f"컨텍스트 길이: {len(full_context):,}자 ({len(full_context)//2:,}토큰 추정)")
    print("=" * 70)
    print(full_context[:2000])
    print("...(truncated)")
    return full_context

# 전체 파이프라인 테스트
context = build_llm_context("K-IFRS 1115", "수행의무 판단 기준", 
                             chunks, ie_results, bc_results)

컨텍스트 길이: 4,602자 (2,301토큰 추정)
# K-IFRS 1115 고객과의 계약에서 생기는 수익
사용자 질문: 수행의무 판단 기준

## 적용 문단 [Authoritative, Level 1]

**문단 B55** (본문, 공시)
B55		라이선스가 구별되지 않는다면, 수행의무(약속한 라이선스 포함)가 기간에 걸쳐 이행되는 수행의무인지 한 시점에 이행되는 수행의무인지를 판단하기 위해 문단 31~38을 적용한다.

**문단 34** (본문, 인식)
34		고객이 자산을 통제하는지를 판단할 때, 그 자산을 재매입하는 약정을 고려한다(문단 B64~B76 참조).
**기간에 걸쳐 이행하는 수행의무**

**문단 14** (본문, 인식)
14		고객과의 계약이 문단 9의 기준을 충족하지 못한다면, 문단 9의 기준이 나중에 충족되는지를 판단하기 위해 그 계약을 지속적으로 검토한다.

**문단 32** (본문, 인식)
32		문단 22~30에 따라 식별한 각 수행의무를 (문단 35~37에 따라) 기간에 걸쳐 이행하는지 또는 (문단 38에 따라) 한 시점에 이행하는지를 계약 개시시점에 판단한다. 수행의무가 기간에 걸쳐 이행되지 않는다면, 그 수행의무는 한 시점에 이행되는 것이다.

**문단 44** (본문, 인식)
44		수행의무의 진행률을 합리적으로 측정할 수 있는 경우에만, 기간에 걸쳐 이행하는 수행의무에 대한 수익을 인식한다. 적절한 진행률 측정방법을 적용하는 데 필요한 신뢰할 수 있는 정보가 부족하다면 수행의무의 진행률을 합리적으로 측정할 수 없을 것이다.

**문단 45** (본문, 인식)
45		어떤 상황(예: 계약 초기 단계)에서는 수행의무의 결과를 합리적으로 측정할 수 없으나, 수행의무를 이행하는 동안에 드는 원가는 회수될 것으로 예상한다. 그 상황에서는 수행의무의 결과를 합리적으로 측정할 수 있을 때까지 발생원가의 범위에서만 수익을 인식한다.

**문단 123** (본문, 공시)
123		이 기준서를 적용하면서 내린, 고객과의 계약에서 생기는 수익

## 추가 테스트 쿼리

In [6]:
def full_pipeline(query: str, include_ie=True, include_bc=True):
    """4-Step 전체 파이프라인 실행"""
    print(f"{'='*70}")
    print(f"QUERY: {query}")
    print(f"{'='*70}\n")
    
    # Step 1
    standards = step1_identify_standard(query)
    selected = standards[0][0]  # top-1
    print(f"\n→ 선택된 기준서: {selected}\n")
    
    # Step 2
    main_chunks, para_nums = step2_search_authoritative(query, selected)
    
    # Step 3 (IE)
    ie = step3_4_find_related(selected, para_nums, "ie") if include_ie else None
    
    # Step 4 (BC)
    bc = step3_4_find_related(selected, para_nums, "bc") if include_bc else None
    
    # 컨텍스트 조합
    print(f"\n{'='*70}")
    print("LLM 컨텍스트:")
    print(f"{'='*70}")
    context = build_llm_context(selected, query, main_chunks, ie, bc)
    
    return context

In [7]:
# 테스트 1: 충당부채 인식 조건 → K-IFRS 1037, 문단 14 기대
ctx = full_pipeline("충당부채 인식 조건")

QUERY: 충당부채 인식 조건

Query: "충당부채 인식 조건"

순위   유사도      기준서                  제목
----------------------------------------------------------------------
1    0.5532   K-IFRS 1037          충당부채 우발부채 우발자산
2    0.4934   K-IFRS 2101          사후처리 및 복구관련 충당부채의 변경
3    0.4641   K-IFRS 2121          부담금
4    0.4479   K-IFRS 1012          법인세
5    0.4458   K-IFRS 2119          지분상품에 의한 금융부채의 소멸

→ 선택된 기준서: K-IFRS 1037

Query: "충당부채 인식 조건" in K-IFRS 1037


### 본문 (main)
  [0.607] 문단 N/A (목적)
         이 기준서의 목적은 충당부채, 우발부채, 우발자산을 회계처리하기 위하여 적절한 인식기준과 측정기준을 마련하고, 재무제표이용자가 충당부채 등의 특성, 발생 시기, 금액을 파악할 수 있도록 충분한 정보를 재무제표 주석에 공시하도록 하는 것이다....
  [0.582] 문단 7 (적용범위)
         7	이 기준서에서는 충당부채를 지출하는 시기 또는 금액이 불확실한 부채로 정의하고 있다. 일부 국가에서는 충당부채라는 용어를 감가상각, 자산손상, 대손 등의 항목과 관련하여 사용하고 있다. 이런 항목은 자산 장부금액의 조정에 해당하며, 이 기준서에서는 다루지 아니한다....
  [0.576] 문단 8 (적용범위)
         8	지출을 자산으로 처리할지 비용으로 처리할지는 다른 한국채택국제회계기준서에서 규정하고 있으며, 이 기준서에서는 다루지 아니한다. 따라서 이 기준서에서는 충당부채의 설정시점에 인식된 원가의 자본화를 금지하거나 요구하지 아니한다....
  [0.568] 문단 9 (적용범위)


In [8]:
# 테스트 2: 리스 식별 → K-IFRS 1116, 문단 9 기대
ctx = full_pipeline("리스 식별")

QUERY: 리스 식별

Query: "리스 식별"

순위   유사도      기준서                  제목
----------------------------------------------------------------------
1    0.3769   K-IFRS 1116          리스
2    0.3139   K-IFRS 1032          금융상품 표시
3    0.3103   K-IFRS 2102          조합원 지분과 유사 지분
4    0.3067   K-IFRS 1117          보험계약
5    0.3038   K-IFRS 1103          사업결합

→ 선택된 기준서: K-IFRS 1116

Query: "리스 식별" in K-IFRS 1116


### 본문 (main)
  [0.585] 문단 N/A (경과 규정(문단 C2~C20))
         **리스의 정의(문단 C3~C4)**...
  [0.568] 문단 N/A (리스제공자)
         **리스의 분류(문단 B53~B58)**...
  [0.553] 문단 N/A (리스이용자)
         **인식**...
  [0.515] 문단 61 (리스제공자)
         **61		리스제공자는 각 리스를 운용리스 아니면 금융리스로 분류한다.**...
  [0.504] 문단 C6 (경과 규정)
         C6		리스이용자는 문단 C5에서 기술하는 선택 사항을 리스이용자에 해당하는 모든 리스에 일관되게 적용한다....
  [0.486] 문단 B12 (판매후리스 거래)
         B12		계약이 잠재적인 별도 리스요소 각각에 대하여 리스를 포함하는지를 판단한다. 별도 리스요소에 대한 지침으로 문단 B32를 참조한다. **	식별되는 자산**...
  [0.485] 문단 88 (리스제공자)
         88		리스제공자는 기초자산의 특성에 따라 재무상태표에 운용리스 대상 기초자산을 표시한다. **공시**...
  [0.47

In [9]:
# 테스트 3: 금융자산 분류 → K-IFRS 1109, 문단 4.1.1~4.1.5 기대
ctx = full_pipeline("금융자산 분류")

QUERY: 금융자산 분류

Query: "금융자산 분류"

순위   유사도      기준서                  제목
----------------------------------------------------------------------
1    0.4414   K-IFRS 1109          금융상품
2    0.4260   K-IFRS 1032          금융상품 표시
3    0.4174   K-IFRS 1107          금융상품 공시
4    0.4166   K-IFRS 1023          차입원가
5    0.4135   K-IFRS 1036          자산손상

→ 선택된 기준서: K-IFRS 1109

Query: "금융자산 분류" in K-IFRS 1109


### 본문 (main)
  [0.857] 문단 N/A (제4장 분류)
         **제4.1절 금융자산의 분류**...
  [0.641] 문단 4.4.2 (제4장 분류)
         **4.4.2	금융부채는 재분류하지 아니한다.**...
  [0.567] 문단 5.6.2 (제5장 측정)
         **5.6.2	금융자산을 상각후원가 측정 범주에서 당기손익-공정가치 측정 범주로 재분류하는 경우에 재분류일의 공정가치로 측정한다. 금융자산의 재분류 전 상각후원가와 공정가치의 차이에 따른 손익은 당기손익으로 인식한다.**...
  [0.565] 문단 4.4.1 (제4장 분류)
         **4.4.1	금융자산을 관리하는 사업모형을 변경하는 경우에만, 영향 받는 모든 금융자산을 문단 4.1.1~4.1.4에 따라 재분류한다. 문단 5.6.1~5.6.7, 문단 B4.4.1~B4.4.3, 문단 B5.6.1~B5.6.2는 금융자산의 재분류에 대한 추가 지침을 ...
  [0.544] 문단 4.1.1 (제4장 분류)
         **4.1.1	문단 4.1.5를 적용하는 경우가 아니라면, 다음 두 가지 사항 모두에 근거하여 금융자산이 후속

In [10]:
# 연결 종료
conn.close()
print("DB 연결 종료")

DB 연결 종료
